# Kaggle | Deleted-Class Reference Diagnostics — Forget 5 / 10 / 15

**Research evaluation notebook** · 4 datasets × 3 scopes = **12 configurations**

This notebook measures the missing quantities from your commented manuscript subsection:

1. **Mean Jensen–Shannon divergence (nats)** between the student and a **matched retrained reference** on **the same forgotten-class test images**.
2. **Mean retrained-model predictive entropy (nats)** on forgotten-class test images.
3. **Frozen-backbone linear-probe accuracy (%)** on forgotten classes for **teacher / retrained / student** (fit probe on forgotten-class *training* examples; test on disjoint forgotten-class *test* examples).
4. **Student-own and retrained-own attack AUC** from independently trained attack models on **retained training members versus retained held-out test nonmembers**. These AUCs are **not deleted-class MIA AUCs**.

### Read this before running

- **Your uploaded `.ipynb` files are source code and saved outputs, not necessarily the original PyTorch weights.** The teacher code did not reliably export teacher weights. Original checkpoints must be attached separately or the notebook can **recreate** models by training them. Recreated weights **are new experimental runs**, **not** the historical weights behind your paper's existing tables.
- **For a publication-matched comparison, provide the original *balanced* train/test datasets** used by the paper. In particular, the paper's Caltech-101 balanced subset has **87 classes**, while the public Kaggle download has **102**. Downloading the public datasets is offered in `public_demo` mode **only for testing the pipeline** and must not be presented as the original matched experiment.
- The notebook **never fills metrics by guesses** and does not reuse the previous Kaggle `demo_first_15` reference weights as if they were matched.
- A fresh missing-model reconstruction can require **far more than one Kaggle session**. Each epoch saves a resumable checkpoint; `TIME_BUDGET_MINUTES=600` means a **10-hour notebook-internal safety window** but **does not extend Kaggle's GPU session or quota**.

### Kaggle setup

1. Import this `.ipynb` at [kaggle.com/code](https://www.kaggle.com/code), enable **GPU T4 ×2** (this code uses one GPU), and turn on **Internet** if ImageNet weights or direct ZIP downloads are needed.
2. **Publication mode (recommended):** Attach the original four balanced ZIPs or extracted folders as a private Kaggle dataset via **Add Input**, or place them in `/kaggle/working` and set `ORIGINAL_INPUT_OVERRIDES` below. Simply attaching the four *public* Kaggle datasets is **not equivalent**.
3. Add saved model `.pth`/`.pt` weights as a Kaggle input dataset if you have them. Configure `EXTERNAL_CHECKPOINTS` when necessary. Otherwise the notebook will train the missing teacher, students and references **as reconstructions**.
4. Run in cell order. After a time-budget pause, use **Save Version → Save & Run All** to persist the `deleted_class_diagnostics` output. On the next session, attach the prior version output through **Add Input** and set `PREVIOUS_RESULTS_DIR` to that folder.
5. Outputs are saved to `/kaggle/working/deleted_class_diagnostics`: CSV + LaTeX table, per-experiment JSON, models, progress checkpoints, and feature caches.

**Interpretation:** Linear probing deliberately trains a new classifier on deleted examples after freezing each backbone. It measures *representation recoverability*, **not** the deleted-class accuracy of the unlearned model itself; the frozen model is not further trained.

In [1]:
# 1. Imports and environment
import os, sys, json, time, random, re, hashlib, shutil, zipfile, tarfile, subprocess, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from IPython.display import display

assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle (kaggle.com/code).'
assert torch.cuda.is_available(), 'Select GPU in Kaggle Settings -> Accelerator.'
DEVICE=torch.device('cuda:0')
print('Torch', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
print('Total GPUs:',torch.cuda.device_count(), '(this notebook deliberately uses only GPU 0)')

Torch 2.11.0+cu128 GPU: Tesla T4
Total GPUs: 2 (this notebook deliberately uses only GPU 0)


## 2 · Configure the data and checkpoints

`original_balanced` (default) enforces the paper's balanced category counts and train/test sample counts. `public_demo` downloads the four Kaggle sources via their **ZIP download API**, not `kagglehub` mounting; it is **diagnostic only** and must not be used to validate your original publication tables.

**Checkpoint policy:** The notebook can auto-retrain missing weights, but the results then pertain to **newly reconstructed models**. If you supply raw `state_dict`s without provenance, the code will reject them unless you deliberately opt in to unverified imports.

In [2]:
# ================== USER CONFIGURATION ==================
DATA_MODE = 'original_balanced'  # 'original_balanced' or 'public_demo'

# For original balanced archives/folders, place originals in /kaggle/input
# via Add Input (recommended), or set explicit paths to your uploaded copies.
ORIGINAL_INPUT_OVERRIDES = {
    # 'Caltech-101': '/kaggle/input/my-balanced-data/caltech balanced dataset1.zip',
    # 'Caltech-256': '/kaggle/input/my-balanced-data/Caltech 256 balanced dataset.zip',
    # 'CIFAR-100': '/kaggle/input/my-balanced-data/cifar100.zip',
    # 'Stanford Dogs': '/kaggle/input/my-balanced-data/Standford dog balanced dataset.zip',
}

# Optional pre-trained teacher, original student, and aligned-reference checkpoints.
# Keys: ('Dataset', 0, 'teacher') or ('Dataset', 5/10/15, 'student'/'reference').
EXTERNAL_CHECKPOINTS = {
    # ('Caltech-101', 5, 'student'): '/kaggle/input/my-models/cal101_forget5_student.pth',
}
ALLOW_UNVERIFIED_EXTERNAL = False  # do not set True for publication claims
AUTO_TRAIN_MISSING_MODELS = True   # train original-like teacher/student + matched references if absent

# Resume: attach the OUTPUT of your previous Kaggle notebook version as an input.
PREVIOUS_RESULTS_DIR = None  # e.g. '/kaggle/input/previous-evaluation/deleted_class_diagnostics'

# Budget 600 min = 10h internal max; cannot change Kaggle's actual runtime/quota.
TIME_BUDGET_MINUTES = 600
RUN_DATASETS = ['Caltech-101', 'Caltech-256', 'CIFAR-100', 'Stanford Dogs']
FORGET_SCOPES = [5, 10, 15]
MAX_NEW_EXPERIMENTS_PER_SESSION = None  # optional 1 or 2 for controlled sessions

IMAGE_SIZE = 224
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 2
TRAIN_SEED = 42
TEACHER_EPOCHS = 30
STUDENT_EPOCHS = 30
REFERENCE_EPOCHS = 30
TEACHER_LR = 1e-4
STUDENT_LR = 3e-4
REFERENCE_LR = 1e-4
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.7
USE_AMP = True

# Frozen-feature probing uses ALL forgotten train/test samples by default.
# MIA per-class caps keep inference manageable on large datasets.
MIA_PAIRS_PER_RETAINED_CLASS = 12
MIA_ATTACK_TEST_FRACTION = 0.30
CACHE_MODEL_FEATURES = True

# CoAtNet defaults are usually ImageNet normalization; verify against the
# exact timm.resolve_model_data_config used in original model runs.
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)

OUTPUT_DIR = Path('/kaggle/working/deleted_class_diagnostics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path('/kaggle/working/deleted_class_dataset_cache')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
START_MONOTONIC = time.monotonic()
DEADLINE = None if TIME_BUDGET_MINUTES is None else START_MONOTONIC + TIME_BUDGET_MINUTES * 60

if PREVIOUS_RESULTS_DIR:
    previous=Path(PREVIOUS_RESULTS_DIR)
    if not previous.is_dir():
        raise FileNotFoundError(f'Previous results not attached: {previous}')
    if previous.resolve() == OUTPUT_DIR.resolve():
        raise ValueError('PREVIOUS_RESULTS_DIR must be a separate saved-version folder.')
    shutil.copytree(previous, OUTPUT_DIR, dirs_exist_ok=True)
    print('Restored saved models and experiment metrics from:', previous)

print('DATA_MODE:', DATA_MODE, '| time budget:', TIME_BUDGET_MINUTES, 'minutes')
print('Output:', OUTPUT_DIR)

DATA_MODE: original_balanced | time budget: 600 minutes
Output: /kaggle/working/deleted_class_diagnostics


## Test the connection first

In [3]:

import socket
import requests

try:
    ip = socket.gethostbyname("www.kaggle.com")
    print("DNS working:", ip)

    response = requests.get(
        "https://www.kaggle.com",
        timeout=15
    )

    print("Kaggle HTTP status:", response.status_code)
    print("Network connection successful.")

except Exception as e:
    print("Network connection failed:", type(e).__name__)
    print(str(e))
    print("Check Settings > Internet.")


DNS working: 35.244.233.98
Kaggle HTTP status: 200
Network connection successful.


## 3 · Dataset discovery and direct ZIP download (only in demo mode)

Original balanced archives are found recursively in Kaggle inputs, with overrides if you supplied nonstandard names. A ZIP is extracted only once. **The notebook will refuse to treat the full public Caltech-101 (102 classes) as equivalent to the original 87-class balanced subset.**

In [4]:

# ============================================================
# SECTION 3 — DIRECT KAGGLE DATASET DOWNLOAD
# PUBLIC DEMO MODE — NO MANUAL DATASET UPLOAD REQUIRED
# ============================================================

DATA_MODE = "public_demo"

from pathlib import Path
import os
import zipfile
import requests
import numpy as np

DATA_ROOT = Path(
    "/kaggle/working/deleted_class_dataset_cache"
)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

RUN_DATASETS = globals().get(
    "RUN_DATASETS",
    ["Caltech-101", "Caltech-256", "CIFAR-100", "Stanford Dogs"]
)

TRAIN_SEED = globals().get("TRAIN_SEED", 42)

DATASET_FOLDERS = {
    "Caltech-101": "Caltech-101",
    "Caltech-256": "Caltech-256",
    "CIFAR-100": "CIFAR-100",
    "Stanford Dogs": "Stanford-Dogs",
}

PUBLIC_KAGGLE_IDS = {
    "Caltech-101": "imbikramsaha/caltech-101",
    "Caltech-256": "jessicali9530/caltech256",
    "CIFAR-100": "melikechan/cifar100",
    "Stanford Dogs": "jessicali9530/stanford-dogs-dataset",
}

EXPECTED_ORIGINAL = {
    "Caltech-101": (87, 2776, 696),
    "Caltech-256": (257, 15420, 5180),
    "CIFAR-100": (100, 50000, 10000),
    "Stanford Dogs": (120, 8400, 3600),
}

IMAGE_EXT = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

# Exact class selections from your original experiments
ORIGINAL_FORGET_15 = {
    "Caltech-101": [
        "trilobite", "brontosaurus", "Leopards",
        "electric_guitar", "dalmatian", "crocodile",
        "camera", "brain", "yin_yang", "rooster",
        "beaver", "stapler", "laptop", "Motorbikes",
        "crayfish"
    ],
    "Caltech-256": [
        "058.doorknob", "013.birdbath",
        "141.microscope", "126.ladder",
        "115.ice-cream-cone", "072.fire-truck",
        "053.desk-globe", "045.computer-keyboard",
        "217.tennis-court", "017.bowling-ball",
        "016.boom-box", "048.conch",
        "112.human-skeleton", "120.joy-stick",
        "014.bilmp"
    ],
    "CIFAR-100": [
        "streetcar", "butterfly", "bear",
        "wardrobe", "girl", "elephant", "cup",
        "castle", "bus", "telephone", "rocket",
        "boy", "skunk", "orchid", "beaver"
    ],
    "Stanford Dogs": [
        "borzoi", "Brittany_spaniel",
        "American_Staffordshire_terrier",
        "giant_schnauzer", "Irish_water_spaniel",
        "Greater_Swiss_Mountain_dog",
        "Gordon_setter", "Chihuahua",
        "Brabancon_griffon", "chow",
        "toy_poodle", "Walker_hound",
        "Boston_bull", "basenji",
        "Rhodesian_ridgeback"
    ]
}


# ============================================================
# DIRECTORY DISCOVERY
# ============================================================

def folders(root, max_depth=7):
    root = Path(root)

    for parent, children, _ in os.walk(root):
        p = Path(parent)

        if len(p.relative_to(root).parts) > max_depth:
            children[:] = []
            continue

        children[:] = [
            c for c in children
            if not c.startswith(".")
        ]

        yield p


def nclasses(root):
    if not root.is_dir():
        return 0

    return sum(
        1 for d in root.iterdir()
        if d.is_dir()
        and any(
            f.is_file()
            and f.suffix.lower() in IMAGE_EXT
            for f in d.iterdir()
        )
    )


def child(root, names):
    return next(
        (
            d for d in root.iterdir()
            if d.is_dir()
            and d.name.lower() in names
        ),
        None
    )


# ============================================================
# SAFE ZIP EXTRACTION
# ============================================================

def extract_zip(archive, target):
    target.mkdir(parents=True, exist_ok=True)
    safe_root = target.resolve()

    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            resolved = (
                safe_root / member.filename
            ).resolve()

            if not resolved.is_relative_to(safe_root):
                raise ValueError(
                    "Unsafe archive filename: "
                    + member.filename
                )

            file_type = (
                member.external_attr >> 16
            ) & 0o170000

            if file_type == 0o120000:
                raise ValueError(
                    "Archive symlink not allowed"
                )

        z.extractall(target)


# ============================================================
# DOWNLOAD DIRECTLY FROM KAGGLE
# ============================================================

def direct_download(name):
    slug = DATASET_FOLDERS[name]

    cached = DATA_ROOT / (
        slug + "_public_download.zip"
    )

    if cached.exists() and zipfile.is_zipfile(cached):
        print("Using cached ZIP:", cached.name)
        return cached

    dataset_id = PUBLIC_KAGGLE_IDS[name]

    url = (
        "https://www.kaggle.com/api/v1/datasets/download/"
        + dataset_id
    )

    temporary = Path(str(cached) + ".part")

    print("Downloading:", dataset_id, flush=True)

    with requests.get(
        url, stream=True, timeout=(30, 240)
    ) as response:

        response.raise_for_status()

        with temporary.open("wb") as output:
            for chunk in response.iter_content(
                chunk_size=2 * 1024 * 1024
            ):
                if chunk:
                    output.write(chunk)

    if not zipfile.is_zipfile(temporary):
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            "Kaggle did not return a valid ZIP. "
            "Enable Internet or check dataset access."
        )

    os.replace(temporary, cached)

    print("Download completed:", cached.name)

    return cached


# ============================================================
# CIFAR-100 BINARY FORMAT
# ============================================================

def find_cifar_binary(base):
    for p in folders(base):
        if all(
            (p / f).is_file()
            for f in ("train", "test", "meta")
        ):
            compat = DATA_ROOT / (
                "CIFAR-100_binary_compat"
            )
            compat.mkdir(
                parents=True, exist_ok=True
            )

            link = compat / "cifar-100-python"

            if link.is_symlink():
                if link.resolve() != p.resolve():
                    link.unlink()

            elif link.exists():
                raise RuntimeError(
                    "Conflicting CIFAR directory"
                )

            if not link.exists():
                link.symlink_to(
                    p.resolve(),
                    target_is_directory=True
                )

            return compat

    return None


# ============================================================
# CLASS NAME NORMALIZATION
# ============================================================

def normalized_name(name, dataset_name):

    # Public Stanford Dogs often uses nXXXXXXXX-Breed.
    if dataset_name == "Stanford Dogs":
        if (
            len(name) > 10
            and name.startswith("n")
            and name[1:9].isdigit()
            and name[9] == "-"
        ):
            return name[10:]

    # Normalize the historical spelling of the blimp class.
    if dataset_name == "Caltech-256":
        if name == "014.blimp":
            return "014.bilmp"

    return name


def link_class_folders(source, destination, dataset_name):
    destination.mkdir(
        parents=True, exist_ok=True
    )

    for cls in source.iterdir():
        if not cls.is_dir():
            continue

        label = normalized_name(
            cls.name, dataset_name
        )

        link = destination / label

        if link.is_symlink():
            if link.resolve() != cls.resolve():
                link.unlink()

        elif link.exists():
            raise RuntimeError(
                "Class directory conflict: " + str(link)
            )

        if not link.exists():
            link.symlink_to(
                cls.resolve(),
                target_is_directory=True
            )

    return destination


# ============================================================
# DETECT OR CREATE TRAIN/TEST SPLITS
# ============================================================

def split_public_folders(base, name):
    roots = list(folders(base))

    # First look for an existing train/test split.
    for root in roots:
        train = child(
            root, {"train", "training"}
        )
        test = child(
            root, {"test", "testing"}
        )

        if train and test:
            if (
                nclasses(train) >= 2
                and nclasses(test) >= 2
            ):
                destination = DATA_ROOT / (
                    DATASET_FOLDERS[name]
                    + "_demo_normalized"
                )

                return (
                    link_class_folders(
                        train,
                        destination / "train",
                        name
                    ),
                    link_class_folders(
                        test,
                        destination / "test",
                        name
                    )
                )

    # Otherwise create a seeded 80/20 split.
    best = max(roots, key=nclasses)

    if nclasses(best) < 2:
        raise RuntimeError(
            f"{name}: No supported class/image folders found."
        )

    destination = DATA_ROOT / (
        DATASET_FOLDERS[name] + "_demo_split"
    )

    marker = destination / ".done"

    if not marker.exists():
        rng = np.random.default_rng(TRAIN_SEED)

        classes = sorted(
            (
                p for p in best.iterdir()
                if p.is_dir()
            ),
            key=lambda p: p.name
        )

        for cls in classes:
            images = sorted(
                p for p in cls.iterdir()
                if p.is_file()
                and p.suffix.lower() in IMAGE_EXT
            )

            if len(images) < 2:
                continue

            label = normalized_name(
                cls.name, name
            )

            count_test = max(
                1, round(len(images) * 0.20)
            )

            test_indices = set(
                rng.permutation(
                    len(images)
                )[:count_test].tolist()
            )

            for i, source in enumerate(images):

                split = (
                    "test"
                    if i in test_indices
                    else "train"
                )

                folder = (
                    destination / split / label
                )
                folder.mkdir(
                    parents=True, exist_ok=True
                )

                link = folder / source.name

                if not link.exists():
                    link.symlink_to(
                        source.resolve()
                    )

        marker.touch()

    return (
        destination / "train",
        destination / "test"
    )


# ============================================================
# DOWNLOAD AND REGISTER ALL FOUR DATASETS
# ============================================================

DATASET_PATHS = {}

print("=" * 65)
print("DIRECT KAGGLE DATASET LOADING")
print("MODE:", DATA_MODE)
print("=" * 65)

for name in RUN_DATASETS:

    print("\n" + "=" * 65)
    print("DATASET:", name)
    print("=" * 65)

    archive = direct_download(name)

    extracted = DATA_ROOT / (
        DATASET_FOLDERS[name] + "_public_extracted"
    )

    marker = extracted / ".complete"

    if not marker.exists():
        print("Extracting:", archive.name)

        extract_zip(
            archive, extracted
        )

        marker.touch()
    else:
        print("Using previously extracted dataset.")

    # Detect CIFAR-100 official Python format.
    binary = (
        find_cifar_binary(extracted)
        if name == "CIFAR-100"
        else None
    )

    if binary is not None:
        train_dir = binary
        test_dir = binary

        print("Official CIFAR-100 format detected.")

    else:
        try:
            train_dir, test_dir = split_public_folders(
                extracted, name
            )

        except RuntimeError:
            # Look for archives inside the downloaded ZIP.
            archives = [
                p for p in extracted.rglob("*")
                if p.is_file()
                and p.suffix.lower() == ".zip"
            ]

            if not archives:
                raise

            for i, inner in enumerate(archives):
                nested = extracted / (
                    "_nested_" + str(i)
                )

                nested_marker = nested / ".done"

                if not nested_marker.exists():
                    print(
                        "Extracting nested archive:",
                        inner.name
                    )

                    extract_zip(inner, nested)
                    nested_marker.touch()

            # Retry dataset detection.
            if name == "CIFAR-100":
                binary = find_cifar_binary(extracted)
            else:
                binary = None

            if binary is not None:
                train_dir = test_dir = binary
            else:
                train_dir, test_dir = split_public_folders(
                    extracted, name
                )

    DATASET_PATHS[name] = (
        train_dir, test_dir
    )

    print("REGISTERED:", name)
    print("Train:", train_dir)
    print("Test :", test_dir)
    print("STATUS: SUCCESS")


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("DATASET REGISTRATION SUMMARY")
print("=" * 65)

for name, paths in DATASET_PATHS.items():
    print(name)
    print(" Train:", paths[0])
    print(" Test :", paths[1])

assert len(DATASET_PATHS) == len(RUN_DATASETS)

print("\n✓ ALL DATASETS REGISTERED")
print("✓ DATA_MODE =", DATA_MODE)
print("✓ CONTINUE TO SECTION 4")

print(
    "\nWARNING: Public-demo results do not reproduce "
    "the original balanced-data experiments."
)


DIRECT KAGGLE DATASET LOADING
MODE: public_demo

DATASET: Caltech-101
Downloading: imbikramsaha/caltech-101
Download completed: Caltech-101_public_download.zip
Extracting: Caltech-101_public_download.zip
REGISTERED: Caltech-101
Train: /kaggle/working/deleted_class_dataset_cache/Caltech-101_demo_split/train
Test : /kaggle/working/deleted_class_dataset_cache/Caltech-101_demo_split/test
STATUS: SUCCESS

DATASET: Caltech-256
Downloading: jessicali9530/caltech256
Download completed: Caltech-256_public_download.zip
Extracting: Caltech-256_public_download.zip
REGISTERED: Caltech-256
Train: /kaggle/working/deleted_class_dataset_cache/Caltech-256_demo_split/train
Test : /kaggle/working/deleted_class_dataset_cache/Caltech-256_demo_split/test
STATUS: SUCCESS

DATASET: CIFAR-100
Downloading: melikechan/cifar100
Download completed: CIFAR-100_public_download.zip
Extracting: CIFAR-100_public_download.zip
REGISTERED: CIFAR-100
Train: /kaggle/working/deleted_class_dataset_cache/CIFAR-100_demo_normalize

## 4 · Verify original classes, forgotten-class identities, train/test splits

**Safety gate:** In original-balanced mode, category counts and sample counts must match those documented in the submitted paper. The forget lists are the original seed-42 class names recovered from the uploaded source notebooks. The code uses **first five / first ten / all fifteen**.

In [5]:
class RawCIFAR100(Dataset):
    # Optional fallback for public-demo CIFAR-100 binary distributions.
    def __init__(self, root, train=True):
        self.base=datasets.CIFAR100(str(root),train=train,download=False)
        original=self.base.classes
        self.classes=sorted(original)
        mapping={name:i for i,name in enumerate(self.classes)}
        self.targets=[mapping[original[y]] for y in self.base.targets]
    def __len__(self): return len(self.base)
    def __getitem__(self,i):
        image,original_label=self.base[i]
        return image,self.targets[i]

RAW_DATA={}
DATASET_MANIFEST={}
FORGET_CLASSES={}
for dataset_name in RUN_DATASETS:
    tr,te=DATASET_PATHS[dataset_name]
    if dataset_name=='CIFAR-100' and (tr/'cifar-100-python'/'train').is_file():
        train_data=RawCIFAR100(tr,train=True)
        test_data=RawCIFAR100(te,train=False)
    else:
        train_data=datasets.ImageFolder(str(tr),transform=None)
        test_data=datasets.ImageFolder(str(te),transform=None)
    class_names=train_data.classes
    if class_names!=test_data.classes:
        raise ValueError(f'{dataset_name} train/test class indexes differ! Stop: cannot safely evaluate.')
    expected=EXPECTED_ORIGINAL[dataset_name]
    actual=(len(class_names),len(train_data),len(test_data))
    if DATA_MODE=='original_balanced' and actual!=expected:
        raise ValueError(f'{dataset_name}: original data should be {expected} (classes,train,test), found {actual}. '
                         'Resolve the balanced dataset/split mismatch before publication.')
    forgotten=ORIGINAL_FORGET_15[dataset_name]
    unknown=[x for x in forgotten if x not in class_names]
    if unknown:
        raise ValueError(f'{dataset_name}: original forgotten categories missing: {unknown}. '
                         'Check dataset version and names; do NOT substitute arbitrary indices.')
    # Manifest independent of the temporary Kaggle extraction root.
    h=hashlib.sha256()
    for split,ds,root in [('train',train_data,tr),('test',test_data,te)]:
        if isinstance(ds,RawCIFAR100):
            h.update(split.encode()); h.update(ds.base.data.tobytes());
            h.update(np.asarray(ds.targets,dtype=np.int32).tobytes())
        else:
            for file_path,label in ds.samples:
                p=Path(file_path)
                h.update(f'{split}|{p.relative_to(root)}|{label}|{p.stat().st_size}\n'.encode())
    fingerprint=h.hexdigest()[:20]
    RAW_DATA[dataset_name]={'train':train_data,'test':test_data,'classes':class_names}
    DATASET_MANIFEST[dataset_name]={'fingerprint':fingerprint,'classes':class_names,
                                     'counts':{'train':len(train_data),'test':len(test_data)},
                                     'paths':[str(tr),str(te)],'data_mode':DATA_MODE}
    for n in FORGET_SCOPES:
        FORGET_CLASSES[(dataset_name,n)]=[class_names.index(cls) for cls in forgotten[:n]]
    print(f'{dataset_name:16} {len(class_names)} classes; {len(train_data)} train; '
          f'{len(test_data)} test | forget-5/10/15: exact names found | id={fingerprint}')

(DATA_ROOT/'manifest.json').write_text(json.dumps(DATASET_MANIFEST,indent=2))
(OUTPUT_DIR/'dataset_manifest.json').write_text(json.dumps(DATASET_MANIFEST,indent=2))
print('Dataset identity and forgotten classes verified.')

Caltech-101      102 classes; 7315 train; 1829 test | forget-5/10/15: exact names found | id=93be4d5f9194f8f55d59
Caltech-256      257 classes; 24490 train; 6117 test | forget-5/10/15: exact names found | id=9f391ee7b983e215020d
CIFAR-100        100 classes; 50000 train; 10000 test | forget-5/10/15: exact names found | id=a242be4dbba6284036f5
Stanford Dogs    120 classes; 16469 train; 4111 test | forget-5/10/15: exact names found | id=e14fad367e652c8fd317
Dataset identity and forgotten classes verified.


## 5 · Define reconstruction protocol and strict checkpoint loading

The **reconstruction** hyperparameters follow the original ResNet-18 teacher/student notebooks as closely as possible: teacher ImageNet-pretrained ResNet-18, Adam `1e-4`, 30 epochs; student randomly initialized ResNet-18, retain-only distillation, Adam `3e-4`, 30 epochs, `T=4`, `α=0.7`. Reference re-training starts from fresh ImageNet-pretrained weights (same prior as the original teacher), 30 epochs of Adam `1e-4` on retained samples only. The original notebook's CoAtNet-derived normalization may be version-dependent; **verify `NORMALIZE_MEAN/STD` against the saved original transform configuration before publication**.

This is a **new reconstruction** when original weights are not supplied, not a retroactive recovery of the originally trained models. The saved checkpoints include dataset hashes and class lists to reject wrong-class weights automatically.

In [6]:
class TimeBudgetPause(Exception): pass

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=True

def transform_for(training=False):
    operations=[transforms.Resize((IMAGE_SIZE,IMAGE_SIZE))]
    if training:
        operations.append(transforms.RandomHorizontalFlip(p=0.5))
    operations += [transforms.ToTensor(),transforms.Normalize(NORMALIZE_MEAN,NORMALIZE_STD)]
    return transforms.Compose(operations)

class View(Dataset):
    def __init__(self,raw,indices,augment=False):
        self.raw=raw; self.indices=np.asarray(indices,dtype=np.int64)
        self.transform=transform_for(augment)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        image,label=self.raw[int(self.indices[i])]
        return self.transform(image.convert('RGB')),int(label)

def get_y(ds):
    return np.asarray(ds.targets,dtype=np.int64)

def loader_view(raw,indices,training=False,batch_size=None,seed=42):
    ds=View(raw,indices,augment=training)
    return DataLoader(ds,batch_size=batch_size or (BATCH_SIZE if training else EVAL_BATCH_SIZE),
                      shuffle=training,num_workers=NUM_WORKERS,pin_memory=True,
                      generator=torch.Generator().manual_seed(seed),persistent_workers=False)

def make_resnet(num_classes, pretrained=False):
    weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model=models.resnet18(weights=weights)
    model.fc=nn.Linear(model.fc.in_features,num_classes)
    return model

def model_slug(dataset_name,scope,role):
    return dataset_name.lower().replace(' ','_').replace('-','_')+f'_forget{scope}_{role}'

def model_metadata(dataset_name,scope,role):
    dataset=DATASET_MANIFEST[dataset_name]
    forgotten=[] if role=='teacher' else [dataset['classes'][i] for i in FORGET_CLASSES[(dataset_name,scope)]]
    params={'role':role,'scope':scope,'dataset':dataset_name,
            'dataset_fingerprint':dataset['fingerprint'],'classes':dataset['classes'],
            'forgotten':forgotten,'seed':TRAIN_SEED,'size':IMAGE_SIZE,
            'normalize_mean':list(NORMALIZE_MEAN),'normalize_std':list(NORMALIZE_STD),
            'initialization':'random' if role=='student' else 'imagenet_resnet18_v1',
            'epochs':{'teacher':TEACHER_EPOCHS,'student':STUDENT_EPOCHS,'reference':REFERENCE_EPOCHS}[role],
            'lr':{'teacher':TEACHER_LR,'student':STUDENT_LR,'reference':REFERENCE_LR}[role],
            'batch_size':BATCH_SIZE,'amp':USE_AMP,
            'protocol':'deleted_diagnostics_v2',
            'student_alpha':KD_ALPHA if role=='student' else None,
            'student_temperature':KD_TEMPERATURE if role=='student' else None}
    params['experiment_id']=hashlib.sha256(json.dumps(params,sort_keys=True).encode()).hexdigest()[:20]
    return params

def paths_for(meta):
    base=OUTPUT_DIR/'checkpoints'
    base.mkdir(exist_ok=True)
    stem=model_slug(meta['dataset'],meta['scope'],meta['role'])+'_'+meta['experiment_id']
    return base/(stem+'.pt'),base/(stem+'.progress.pt')

def load_checkpoint(path,meta,allow_unverified=False):
    payload=torch.load(str(path),map_location='cpu',weights_only=True)
    if isinstance(payload,dict) and 'state_dict' in payload:
        state=payload['state_dict'];existing=payload.get('metadata',{})
    elif isinstance(payload,dict) and 'model_state_dict' in payload:
        state=payload['model_state_dict'];existing=payload.get('metadata',{})
    elif isinstance(payload,dict) and all(isinstance(v,torch.Tensor) for v in payload.values()):
        state=payload;existing={}
    else: raise ValueError(f'Unknown checkpoint format: {path}')
    if existing.get('experiment_id')!=meta['experiment_id']:
        if not allow_unverified:
            raise ValueError(f'Checkpoint {path} has missing/mismatched manifest ID. '
                             'Do not compare mismatched models. Use a verified checkpoint or explicitly opt into unverified import.')
        warnings.warn('Loading unverified EXTERNAL weights: do not describe as protocol-matched.')
    clean={k.removeprefix('module.'):v for k,v in state.items()}
    model=make_resnet(len(meta['classes']),pretrained=False)
    model.load_state_dict(clean,strict=True)
    if existing.get('experiment_id') != meta['experiment_id']:
        provenance='unverified_external'
    elif payload.get('origin')=='reconstructed':
        provenance='reconstructed'
    elif payload.get('origin')=='original_saved':
        provenance='verified_original'
    else:
        provenance='verified_identity_unknown_origin'
    return model.to(DEVICE),provenance

def output_checkpoint(dataset_name,scope,role):
    meta=model_metadata(dataset_name,scope,role)
    final,progress=paths_for(meta)
    if final.is_file():
        model,provenance=load_checkpoint(final,meta)
        return model,meta,final,provenance
    key=(dataset_name,scope,role)
    if key in EXTERNAL_CHECKPOINTS:
        path=Path(EXTERNAL_CHECKPOINTS[key])
        if not path.is_file(): raise FileNotFoundError(path)
        model,provenance=load_checkpoint(path,meta,allow_unverified=ALLOW_UNVERIFIED_EXTERNAL)
        return model,meta,path,provenance
    if not AUTO_TRAIN_MISSING_MODELS:
        raise FileNotFoundError(f'Missing {key}. Supply checkpoint in EXTERNAL_CHECKPOINTS or allow training.')
    model=train_missing_model(dataset_name,scope,role,meta,final,progress)
    return model,meta,final,'reconstructed'

print('Model metadata and strict checkpoint validation ready.')

Model metadata and strict checkpoint validation ready.


## 6 · Train missing models, with **checkpoint after every epoch**

Training is intentionally **reproducible reconstruction**, not a guarantee of reproducing historical model weights. A time pause is raised only **after saving an epoch checkpoint**. Restore the previous Kaggle output to resume rather than recomputing. Teacher is shared across the three scopes for each dataset.

In [7]:
def train_missing_model(name,scope,role,meta,final,progress):
    seed=int(TRAIN_SEED)+(scope*101 if role!='teacher' else 0)+{'teacher':0,'student':7,'reference':13}[role]
    seed_all(seed)
    raw=RAW_DATA[name]['train']
    labels=get_y(raw)
    removed=set(FORGET_CLASSES[(name,scope)]) if role!='teacher' else set()
    indices=np.where(~np.isin(labels,list(removed)))[0]
    if len(indices)==0: raise ValueError('No retained training samples!')
    train_loader=loader_view(raw,indices,training=True,seed=seed)

    # Use an ImageNet starting backbone for full teacher and retrained reference.
    model=make_resnet(len(meta['classes']),pretrained=(role!='student')).to(DEVICE)
    teacher=None
    if role=='student':
        teacher,tmeta,tfile,tprov=output_checkpoint(name,0,'teacher')
        teacher.eval()
        for p in teacher.parameters():p.requires_grad_(False)
    lr={'teacher':TEACHER_LR,'student':STUDENT_LR,'reference':REFERENCE_LR}[role]
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    amp=bool(USE_AMP and DEVICE.type=='cuda')
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1
    history=[]
    if progress.exists():
        previous=torch.load(progress,map_location='cpu',weights_only=True)
        if previous.get('experiment_id')!=meta['experiment_id']:
            raise ValueError(f'Progress checkpoint mismatch: {progress}')
        model.load_state_dict(previous['model_state_dict'])
        opt.load_state_dict(previous['optimizer_state_dict'])
        scaler.load_state_dict(previous['scaler_state_dict'])
        history=previous.get('history',[])
        start_epoch=int(previous['epoch'])+1
        print('Resuming',name,role,scope,'at epoch',start_epoch,flush=True)
    epochs=meta['epochs']
    print(f'BEGIN {name} | {role} | Forget-{scope} | N-train={len(indices)} '
          f'| epochs {start_epoch}-{epochs}',flush=True)
    for epoch in range(start_epoch,epochs+1):
        model.train()
        t0=time.monotonic(); total=0;loss_total=0.
        for xb,yb in train_loader:
            xb=xb.to(DEVICE,non_blocking=True);yb=yb.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',enabled=amp):
                logits=model(xb)
                if role=='student':
                    with torch.no_grad():
                        target=teacher(xb)
                    soft=F.kl_div(F.log_softmax(logits/KD_TEMPERATURE,dim=1),
                                  F.softmax(target/KD_TEMPERATURE,dim=1),
                                  reduction='batchmean')*(KD_TEMPERATURE**2)
                    loss=KD_ALPHA*soft+(1-KD_ALPHA)*F.cross_entropy(logits,yb)
                else:
                    loss=F.cross_entropy(logits,yb)
            scaler.scale(loss).backward()
            scaler.step(opt);scaler.update()
            n=len(yb);total+=n;loss_total+=float(loss.detach().cpu())*n
        record={'epoch':epoch,'loss':round(loss_total/max(total,1),6),
                'epoch_seconds':round(time.monotonic()-t0,1)}
        history.append(record)
        payload={'experiment_id':meta['experiment_id'],'epoch':epoch,
                 'model_state_dict':model.state_dict(),
                 'optimizer_state_dict':opt.state_dict(),
                 'scaler_state_dict':scaler.state_dict(),'history':history}
        tmp=progress.with_name(progress.name+'.tmp')
        torch.save(payload,tmp); os.replace(tmp,progress)
        print(f"  epoch {epoch:2}/{epochs}: loss={record['loss']:.4f}, "
              f"time={record['epoch_seconds']:.1f}s",flush=True)
        if DEADLINE is not None and time.monotonic()>=DEADLINE and epoch<epochs:
            raise TimeBudgetPause(f'{name} {role} Forget-{scope} safely paused after epoch '
                                  f'{epoch}/{epochs}; checkpoint saved.')
    result={'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'metadata':meta,'origin':'reconstructed','history':history}
    tmp=final.with_name(final.name+'.tmp')
    torch.save(result,tmp);os.replace(tmp,final)
    progress.unlink(missing_ok=True)
    pd.DataFrame(history).to_csv(final.with_suffix('.history.csv'),index=False)
    if teacher is not None: del teacher
    print('MODEL SAVED:',final,flush=True)
    return model

print('Epoch-resumable teacher/student/reference training functions ready.')

Epoch-resumable teacher/student/reference training functions ready.


## 7 · Inference caches, JS divergence, entropy, deleted-class linear probes, independent MIA attacks

All metrics use identical sample indices for matched student/reference pairs. A new linear classifier is trained *after* the frozen features are extracted, using forgotten-class training images, then tested on never-used forgotten-class test images. Own-attack AUC is calculated **separately per model** on retained training vs retained test examples, with a disjoint attack training/testing split. Attack populations are class-balanced and fixed across models.

In [8]:
def select_indices(name,scope):
    train_y=get_y(RAW_DATA[name]['train'])
    test_y=get_y(RAW_DATA[name]['test'])
    classes=np.asarray(FORGET_CLASSES[(name,scope)],dtype=np.int64)
    return {
      'forget_train':np.flatnonzero(np.isin(train_y,classes)),
      'forget_test':np.flatnonzero(np.isin(test_y,classes)),
      'retain_train':np.flatnonzero(~np.isin(train_y,classes)),
      'retain_test':np.flatnonzero(~np.isin(test_y,classes)),
    }

def mia_pairs(name,scope):
    # The SAME per-class sampling is used to evaluate each role.
    ytr=get_y(RAW_DATA[name]['train']); yte=get_y(RAW_DATA[name]['test'])
    forgotten=set(FORGET_CLASSES[(name,scope)])
    rng=np.random.default_rng(TRAIN_SEED+scope*37)
    members=[];nonmembers=[]
    for c in range(len(RAW_DATA[name]['classes'])):
        if c in forgotten:continue
        tr=np.flatnonzero(ytr==c);te=np.flatnonzero(yte==c)
        n=min(len(tr),len(te),MIA_PAIRS_PER_RETAINED_CLASS)
        if n<4: continue
        members.extend(rng.choice(tr,n,replace=False).tolist())
        nonmembers.extend(rng.choice(te,n,replace=False).tolist())
    if len(members)<20:raise ValueError(f'Insufficient retained members/nonmembers: {name} {scope}')
    return np.asarray(members,dtype=np.int64),np.asarray(nonmembers,dtype=np.int64)

@torch.inference_mode()
def compute_outputs(model,raw,indices):
    hook_features=[]
    def before_fc(mod,inp):hook_features.append(inp[0].detach().cpu().numpy())
    handle=model.fc.register_forward_pre_hook(before_fc)
    p_list=[];y_list=[];f_list=[]
    try:
        model.eval()
        loader=loader_view(raw,indices,training=False,seed=TRAIN_SEED)
        for xb,yb in loader:
            xb=xb.to(DEVICE,non_blocking=True)
            hook_features.clear()
            logits=model(xb)
            probs=F.softmax(logits,dim=1).float().cpu().numpy()
            if len(hook_features)!=1:raise AssertionError('ResNet-18 feature hook failed.')
            p_list.append(probs)
            f_list.append(hook_features[0])
            y_list.append(yb.numpy())
    finally:handle.remove()
    return {'probs':np.concatenate(p_list),
            'features':np.concatenate(f_list),
            'labels':np.concatenate(y_list)}

def cached_outputs(name,scope,role,model,meta,subset,indices):
    digest=hashlib.sha256(indices.astype('<i8').tobytes()).hexdigest()[:16]
    filename=f'{model_slug(name,scope,role)}_{meta["experiment_id"]}_{subset}_{digest}.npz'
    cache_dir=OUTPUT_DIR/'feature_caches';cache_dir.mkdir(exist_ok=True)
    cache=cache_dir/filename
    if CACHE_MODEL_FEATURES and cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            return {key:z[key] for key in ('probs','features','labels')}
    split='train' if subset.endswith('_train') else 'test'
    answer=compute_outputs(model,RAW_DATA[name][split],indices)
    if CACHE_MODEL_FEATURES:
        temp=cache.with_name(cache.name+'.tmp')
        with temp.open('wb') as fh:
            np.savez(fh,**answer)
        os.replace(temp,cache)
    return answer

def mean_jsd(p,q):
    if p.shape!=q.shape:raise ValueError(f'Paired probabilities differ: {p.shape} vs {q.shape}')
    eps=1e-12
    p=np.clip(p.astype('float64'),eps,1.);q=np.clip(q.astype('float64'),eps,1.)
    m=.5*(p+q)
    per=.5*np.sum(p*np.log(p/m),axis=1)+.5*np.sum(q*np.log(q/m),axis=1)
    return float(np.mean(per))

def entropy_nats(p):
    p=np.clip(p.astype('float64'),1e-12,1.)
    return float(np.mean(-np.sum(p*np.log(p),axis=1)))

def probe_accuracy(train_outputs,test_outputs):
    ytr=train_outputs['labels'];yte=test_outputs['labels']
    if len(np.unique(ytr))<2 or not set(np.unique(yte)).issubset(set(ytr)):
        raise ValueError('Linear probe requires >=2 deleted classes represented in both partitions.')
    model=make_pipeline(StandardScaler(),RidgeClassifier(alpha=1.0))
    model.fit(train_outputs['features'],ytr)
    return 100*accuracy_score(yte,model.predict(test_outputs['features']))

def attack_features(out):
    p=out['probs'].astype('float64');y=out['labels']
    p=np.clip(p,1e-12,1.)
    top=np.sort(p,axis=1)[:,-min(3,p.shape[1]):]
    top=np.pad(top,((0,0),(max(0,3-top.shape[1]),0)),constant_values=0)
    highest=top[:,-1];second=top[:,-2]
    truth=p[np.arange(len(y)),y]
    ent=-np.sum(p*np.log(p),axis=1)
    return np.column_stack([top,highest-second,ent,truth,-np.log(truth)])

def own_attack_auc(train_outputs,test_outputs):
    xm=attack_features(train_outputs)
    xn=attack_features(test_outputs)
    features=np.concatenate([xm,xn],axis=0)
    labels=np.r_[np.ones(len(xm),dtype=int),np.zeros(len(xn),dtype=int)]
    idx=np.arange(len(labels))
    itr,ite=train_test_split(idx,test_size=MIA_ATTACK_TEST_FRACTION,
                             random_state=TRAIN_SEED,stratify=labels)
    classifier=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,
                                         class_weight='balanced',random_state=TRAIN_SEED))
    classifier.fit(features[itr],labels[itr])
    scores=classifier.predict_proba(features[ite])[:,1]
    return float(roc_auc_score(labels[ite],scores))

print('Metric functions ready: JSD, entropy, 3 probes and 2 own-MIA AUCs.')

Metric functions ready: JSD, entropy, 3 probes and 2 own-MIA AUCs.


## 8 · Execute the 12 evaluations (time-budgeted; save after every completed configuration)

This stage trains missing weights when necessary, computes the evaluation metrics, and saves a JSON + CSV after **each completed experiment**. A paused epoch remains resumable, and partial tables remain available. Results from the former `demo_first_15` Kaggle notebook are never imported as matching evidence.

In [9]:
SUMMARY_FILE=OUTPUT_DIR/'deleted_class_missing_metrics_12.csv'
ERRORS_FILE=OUTPUT_DIR/'errors.csv'

def experiment_id(name,scope):
    inp={'dataset':name,'scope':scope,'manifest':DATASET_MANIFEST[name]['fingerprint'],
         'classes':[RAW_DATA[name]['classes'][i] for i in FORGET_CLASSES[(name,scope)]],
         'models':{r:model_metadata(name,0 if r=='teacher' else scope,r)['experiment_id']
                   for r in ('teacher','student','reference')},
         'mia_pairs_per_class':MIA_PAIRS_PER_RETAINED_CLASS,
         'mia_split':MIA_ATTACK_TEST_FRACTION,
         'algorithm':'diagnostics_v2'}
    return hashlib.sha256(json.dumps(inp,sort_keys=True).encode()).hexdigest()[:20]

def evaluate_one(name,scope):
    selection=select_indices(name,scope)
    miam, mian=mia_pairs(name,scope)
    if any(len(selection[x])==0 for x in ('forget_train','forget_test','retain_train','retain_test')):
        raise ValueError('Selected forget/retain train/test partition empty')
    models_loaded={};metas={};sources={};provenance={}
    for role in ('teacher','student','reference'):
        use_scope=0 if role=='teacher' else scope
        mod,meta,path,origin=output_checkpoint(name,use_scope,role)
        models_loaded[role]=mod
        metas[role]=meta;sources[role]=str(path);provenance[role]=origin
        print('MODEL',role,origin,path,flush=True)

    if DEADLINE is not None and time.monotonic()>=DEADLINE:
        raise TimeBudgetPause('All model checkpoints are saved; evaluation will resume next session.')
    role_outputs={}
    try:
        for role,model in models_loaded.items():
            use_scope=0 if role=='teacher' else scope
            meta=metas[role]
            role_outputs[role]={
                'forget_train':cached_outputs(name,use_scope,role,model,meta,'forget_train',selection['forget_train']),
                'forget_test':cached_outputs(name,use_scope,role,model,meta,'forget_test',selection['forget_test']),
            }
            if role!='teacher':
                role_outputs[role]['mia_train']=cached_outputs(name,use_scope,role,model,meta,'mia_train',miam)
                role_outputs[role]['mia_test']=cached_outputs(name,use_scope,role,model,meta,'mia_test',mian)
    finally:
        for model in models_loaded.values(): del model
        models_loaded.clear()
        torch.cuda.empty_cache()

    s=role_outputs['student'];r=role_outputs['reference'];t=role_outputs['teacher']
    if not np.array_equal(s['forget_test']['labels'],r['forget_test']['labels']):
        raise AssertionError('Student and reference forgotten-test order is not identical')
    row={
        'Dataset':name,'Forget scope':f'{scope} classes',
        'JS divergence (nats)':round(mean_jsd(s['forget_test']['probs'],r['forget_test']['probs']),6),
        'Retrained entropy (nats)':round(entropy_nats(r['forget_test']['probs']),6),
        'Teacher linear probe (%)':round(probe_accuracy(t['forget_train'],t['forget_test']),4),
        'Retrained linear probe (%)':round(probe_accuracy(r['forget_train'],r['forget_test']),4),
        'Student linear probe (%)':round(probe_accuracy(s['forget_train'],s['forget_test']),4),
        'Student-own MIA AUC (retain)':round(own_attack_auc(s['mia_train'],s['mia_test']),6),
        'Retrained-own MIA AUC (retain)':round(own_attack_auc(r['mia_train'],r['mia_test']),6),
        'N forgotten train':len(selection['forget_train']),
        'N forgotten test':len(selection['forget_test']),
        'N attack members':len(miam),'N attack nonmembers':len(mian),
        'Experiment ID':experiment_id(name,scope),
        'Dataset mode':DATA_MODE,'Dataset fingerprint':DATASET_MANIFEST[name]['fingerprint'],
        'Teacher provenance':provenance['teacher'],
        'Student provenance':provenance['student'],
        'Reference provenance':provenance['reference'],
        'Paper matched':bool(DATA_MODE=='original_balanced' and all(x=='verified_original' for x in provenance.values())),
        'Forgotten classes':json.dumps([RAW_DATA[name]['classes'][i] for i in FORGET_CLASSES[(name,scope)]]),
        'Checkpoint paths':json.dumps(sources),
    }
    detail_dir=OUTPUT_DIR/'individual_metrics';detail_dir.mkdir(exist_ok=True)
    path=detail_dir/(model_slug(name,scope,'metrics')+'_'+row['Experiment ID']+'.json')
    path.write_text(json.dumps(row,indent=2))
    return row

def run_all_metrics():
    rows=(pd.read_csv(SUMMARY_FILE).to_dict('records') if SUMMARY_FILE.exists() else [])
    completed={(r.get('Dataset'),r.get('Forget scope'),str(r.get('Experiment ID')))
               for r in rows}
    completed_new=0
    for name in RUN_DATASETS:
        for scope in FORGET_SCOPES:
            key=(name,f'{scope} classes',experiment_id(name,scope))
            if key in completed:
                print('SKIP saved configuration',name,scope,flush=True);continue
            if DEADLINE is not None and time.monotonic()>=DEADLINE:
                print('TIME-BUDGET PAUSE: no further experiments started.');return pd.DataFrame(rows)
            if MAX_NEW_EXPERIMENTS_PER_SESSION is not None and completed_new>=MAX_NEW_EXPERIMENTS_PER_SESSION:
                print('CONFIGURATION LIMIT: save results and resume next session.');return pd.DataFrame(rows)
            print('\n'+'='*68+'\nEVALUATE',name,'Forget',scope,flush=True)
            try:
                result=evaluate_one(name,scope)
                rows=[r for r in rows if not (r.get('Dataset')==name and r.get('Forget scope')==f'{scope} classes')]
                rows.append(result)
                pd.DataFrame(rows).to_csv(SUMMARY_FILE,index=False)
                completed_new+=1
                print('SAVED completed metrics',name,scope,'|',SUMMARY_FILE,flush=True)
                display(pd.DataFrame([result]).iloc[:,:9])
            except TimeBudgetPause as exc:
                print('SAFE PAUSE:',exc,flush=True)
                return pd.DataFrame(rows)
            except Exception as exc:
                failed={'Dataset':name,'Forget scope':scope,'Error':repr(exc),'When':time.time()}
                old=pd.read_csv(ERRORS_FILE) if ERRORS_FILE.exists() else pd.DataFrame()
                pd.concat([old,pd.DataFrame([failed])],ignore_index=True).to_csv(ERRORS_FILE,index=False)
                raise
    return pd.DataFrame(rows)

results=run_all_metrics()
print('\nCompleted rows:',len(results),'of',len(RUN_DATASETS)*len(FORGET_SCOPES))
print('CSV:',SUMMARY_FILE)


EVALUATE Caltech-101 Forget 5
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s] 


BEGIN Caltech-101 | teacher | Forget-0 | N-train=7315 | epochs 1-30
  epoch  1/30: loss=1.4848, time=37.3s
  epoch  2/30: loss=0.3409, time=14.9s
  epoch  3/30: loss=0.1476, time=14.6s
  epoch  4/30: loss=0.0733, time=15.2s
  epoch  5/30: loss=0.0595, time=14.9s
  epoch  6/30: loss=0.0404, time=15.2s
  epoch  7/30: loss=0.0324, time=15.0s
  epoch  8/30: loss=0.0365, time=15.4s
  epoch  9/30: loss=0.0369, time=15.2s
  epoch 10/30: loss=0.0367, time=15.0s
  epoch 11/30: loss=0.0403, time=14.9s
  epoch 12/30: loss=0.0195, time=14.8s
  epoch 13/30: loss=0.0415, time=15.1s
  epoch 14/30: loss=0.0331, time=15.1s
  epoch 15/30: loss=0.0274, time=14.7s
  epoch 16/30: loss=0.0155, time=15.0s
  epoch 17/30: loss=0.0208, time=15.0s
  epoch 18/30: loss=0.0144, time=14.9s
  epoch 19/30: loss=0.0376, time=15.1s
  epoch 20/30: loss=0.0277, time=15.3s
  epoch 21/30: loss=0.0204, time=15.0s
  epoch 22/30: loss=0.0141, time=14.8s
  epoch 23/30: loss=0.0083, time=14.8s
  epoch 24/30: loss=0.0289, time=15

,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,Caltech-101,5 classes,0.427596,0.986732,100.0,98.9362,95.7447,0.799715,0.667244



EVALUATE Caltech-101 Forget 10
MODEL teacher reconstructed /kaggle/working/deleted_class_diagnostics/checkpoints/caltech_101_forget0_teacher_5b42508a1ee382a56612.pt
BEGIN Caltech-101 | student | Forget-10 | N-train=6693 | epochs 1-30
  epoch  1/30: loss=17.0151, time=24.2s
  epoch  2/30: loss=14.2077, time=16.3s
  epoch  3/30: loss=12.6028, time=16.6s
  epoch  4/30: loss=11.3483, time=16.3s
  epoch  5/30: loss=10.1951, time=16.2s
  epoch  6/30: loss=9.1513, time=16.2s
  epoch  7/30: loss=8.2630, time=16.3s
  epoch  8/30: loss=7.3765, time=16.3s
  epoch  9/30: loss=6.5354, time=16.3s
  epoch 10/30: loss=5.9282, time=16.3s
  epoch 11/30: loss=5.1764, time=16.2s
  epoch 12/30: loss=4.6593, time=16.2s
  epoch 13/30: loss=4.1292, time=16.2s
  epoch 14/30: loss=3.6467, time=16.3s
  epoch 15/30: loss=3.2599, time=16.4s
  epoch 16/30: loss=2.7985, time=16.3s
  epoch 17/30: loss=2.6102, time=16.3s
  epoch 18/30: loss=2.2142, time=16.3s
  epoch 19/30: loss=2.0413, time=16.3s
  epoch 20/30: loss

...


EVALUATE CIFAR-100 Forget 5
BEGIN CIFAR-100 | teacher | Forget-0 | N-train=50000 | epochs 1-30
  epoch  1/30: loss=1.6005, time=96.4s
  epoch  2/30: loss=0.8420, time=96.4s
  epoch  3/30: loss=0.6188, time=95.5s
  epoch  4/30: loss=0.4727, time=95.7s
  epoch  5/30: loss=0.3688, time=95.7s
  epoch  6/30: loss=0.2900, time=95.9s
  epoch  7/30: loss=0.2427, time=95.8s
  epoch  8/30: loss=0.1988, time=96.8s
  epoch  9/30: loss=0.1723, time=96.1s
  epoch 10/30: loss=0.1588, time=96.3s
  epoch 11/30: loss=0.1347, time=96.2s
  epoch 12/30: loss=0.1208, time=96.5s
  epoch 13/30: loss=0.1206, time=96.4s
  epoch 14/30: loss=0.1073, time=96.3s
  epoch 15/30: loss=0.1035, time=96.1s
  epoch 16/30: loss=0.0921, time=96.2s
  epoch 17/30: loss=0.0904, time=96.1s
  epoch 18/30: loss=0.0859, time=96.1s
  epoch 19/30: loss=0.0810, time=96.7s
  epoch 20/30: loss=0.0749, time=96.4s
  epoch 21/30: loss=0.0735, time=96.5s
  epoch 22/30: loss=0.0709, time=95.8s
  epoch 23/30: loss=0.0700, time=96.6s
  epoch

,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,CIFAR-100,5 classes,0.348996,0.671663,98.4,97.8,96.4,0.714587,0.686369



EVALUATE CIFAR-100 Forget 10
MODEL teacher reconstructed /kaggle/working/deleted_class_diagnostics/checkpoints/cifar_100_forget0_teacher_1375155f42975480e770.pt
BEGIN CIFAR-100 | student | Forget-10 | N-train=45000 | epochs 1-30
  epoch  1/30: loss=26.2510, time=119.4s
  epoch  2/30: loss=20.2748, time=110.1s
  epoch  3/30: loss=15.9771, time=109.3s
  epoch  4/30: loss=13.3089, time=109.2s
  epoch  5/30: loss=11.3788, time=109.3s
  epoch  6/30: loss=10.0270, time=109.5s
  epoch  7/30: loss=8.8229, time=109.3s
  epoch  8/30: loss=7.8549, time=109.0s
  epoch  9/30: loss=7.0272, time=109.3s
  epoch 10/30: loss=6.2775, time=109.4s
  epoch 11/30: loss=5.6506, time=109.1s
  epoch 12/30: loss=5.0397, time=109.2s
  epoch 13/30: loss=4.5349, time=109.0s
  epoch 14/30: loss=4.0585, time=109.1s
  epoch 15/30: loss=3.6717, time=109.4s
  epoch 16/30: loss=3.3391, time=109.9s
  epoch 17/30: loss=3.0717, time=109.7s
  epoch 18/30: loss=2.8413, time=111.4s
  epoch 19/30: loss=2.6129, time=112.4s
  ep

,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,CIFAR-100,10 classes,0.346571,0.651468,93.3,88.1,87.6,0.740055,0.702875



EVALUATE CIFAR-100 Forget 15
MODEL teacher reconstructed /kaggle/working/deleted_class_diagnostics/checkpoints/cifar_100_forget0_teacher_1375155f42975480e770.pt
BEGIN CIFAR-100 | student | Forget-15 | N-train=42500 | epochs 1-30
  epoch  1/30: loss=26.1866, time=117.2s
  epoch  2/30: loss=20.6220, time=107.0s
  epoch  3/30: loss=16.3259, time=106.4s
  epoch  4/30: loss=13.5459, time=106.9s
  epoch  5/30: loss=11.6422, time=106.7s
  epoch  6/30: loss=10.1561, time=106.8s
  epoch  7/30: loss=9.0076, time=107.2s
  epoch  8/30: loss=7.9966, time=106.3s
  epoch  9/30: loss=7.1353, time=106.4s
  epoch 10/30: loss=6.4094, time=106.6s
  epoch 11/30: loss=5.7378, time=106.8s
  epoch 12/30: loss=5.1200, time=106.4s
  epoch 13/30: loss=4.5493, time=106.4s
  epoch 14/30: loss=4.1044, time=106.8s
  epoch 15/30: loss=3.7410, time=106.7s
  epoch 16/30: loss=3.3724, time=106.8s
  epoch 17/30: loss=3.1248, time=106.9s
  epoch 18/30: loss=2.8607, time=106.5s
  epoch 19/30: loss=2.6565, time=107.0s
  ep

## 9 · Export formatted 12-row table (CSV + LaTeX)

**A number is reported only if that experiment finished.** Blank cells mean that the required models or metrics were not evaluated; they are never interpolated.

In [10]:
METRICS=['JS divergence (nats)','Retrained entropy (nats)',
         'Teacher linear probe (%)','Retrained linear probe (%)','Student linear probe (%)',
         'Student-own MIA AUC (retain)','Retrained-own MIA AUC (retain)']
all_rows=pd.DataFrame([{'Dataset':name,'Forget scope':f'{s} classes'}
                       for name in RUN_DATASETS for s in FORGET_SCOPES])
if SUMMARY_FILE.exists():
    df=pd.read_csv(SUMMARY_FILE)
    # Keep ONLY rows with the current data+model identities.
    expected={(name,f'{s} classes'):experiment_id(name,s)
              for name in RUN_DATASETS for s in FORGET_SCOPES}
    df=df[df.apply(lambda r:str(r.get('Experiment ID'))==expected.get((r['Dataset'],r['Forget scope']),''),axis=1)]
    df=df.drop_duplicates(['Dataset','Forget scope'],keep='last')
    table=all_rows.merge(df[['Dataset','Forget scope']+METRICS],on=['Dataset','Forget scope'],how='left')
else:
    table=all_rows.copy()
    for col in METRICS:table[col]=np.nan

print('COMPLETED:',table[METRICS[0]].notna().sum(),'/ 12')
display(table.style.format({col:'{:.4f}' for col in METRICS},na_rep='—'))
output=OUTPUT_DIR/'deleted_class_diagnostics_paper_table.csv'
table.to_csv(output,index=False)
valid_provenance=(SUMMARY_FILE.exists() and len(pd.read_csv(SUMMARY_FILE))==12
                  and pd.read_csv(SUMMARY_FILE)['Paper matched'].astype(str).str.lower().eq('true').all())
if not valid_provenance:
    print('WARNING: One or more results use recreated/unverified weights or missing data. '
          'Do NOT describe this as a verified historical-model comparison.')
latex=table.to_latex(index=False,escape=True,na_rep='---',float_format='%.4f',
  caption='Deleted-class diagnostic results (reconstructed or independently verified provenance '
          'must be documented separately). Own-MIA AUC uses retained train/test membership.',
  label='tab:deleted_class_retrained_diagnostics')
(OUTPUT_DIR/'deleted_class_diagnostics_paper_table.tex').write_text(latex)
print('Saved:',output)
print('Saved:',OUTPUT_DIR/'deleted_class_diagnostics_paper_table.tex')
if table[METRICS[0]].notna().sum()==12:
    print('All rows are computed. BEFORE SUBMISSION, inspect checkpoint provenance and Paper matched flags.')
else:
    print('Partial table only: resume remaining model training/evaluations from checkpoints.')

COMPLETED: 8 / 12


,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,Caltech-101,5 classes,0.4276,0.9867,100.0000,98.9362,95.7447,0.7997,0.6672
1,Caltech-101,10 classes,0.4859,1.1603,98.7179,92.3077,87.8205,0.8146,0.6636
2,Caltech-101,15 classes,0.4111,0.9846,99.4505,97.2527,92.5824,0.7784,0.6987
3,Caltech-256,5 classes,0.5356,1.2425,85.1562,75.0000,60.1562,0.8329,0.7424
4,Caltech-256,10 classes,0.4831,1.2567,90.7489,82.3789,73.1278,0.8300,0.7424
5,Caltech-256,15 classes,0.5049,1.2682,89.5706,83.4356,69.9387,0.8331,0.7579
6,CIFAR-100,5 classes,0.3490,0.6717,98.4000,97.8000,96.4000,0.7146,0.6864
7,CIFAR-100,10 classes,0.3466,0.6515,93.3000,88.1000,87.6000,0.7401,0.7029
8,CIFAR-100,15 classes,—,—,—,—,—,—,—
9,Stanford Dogs,5 classes,—,—,—,—,—,—,—


Saved: /kaggle/working/deleted_class_diagnostics/deleted_class_diagnostics_paper_table.csv
Saved: /kaggle/working/deleted_class_diagnostics/deleted_class_diagnostics_paper_table.tex
Partial table only: resume remaining model training/evaluations from checkpoints.


## 10 · Publication checklist and interpretation

- **Check dataset origin**: The 87-class balanced Caltech-101 subset and the original balanced splits must be used. `public_demo` output is not a publication-equivalent baseline.
- **Check model provenance**: For original-paper claims, the checkpoints must correspond to the original teacher/student runs, the same trained-reference protocol, and the same class mapping. Models created by this notebook are a **new reconstruction experiment** and can be reported as such, but **must not be passed off as historical saved weights**.
- **Check transformations**: The original teacher/student source used `timm` CoAtNet model-data normalization in its later loader setup. Confirm the exact numeric means/standard deviations and source training epochs before claiming exact reproduction.
- **What each measurement means**: JS divergence compares full class-distribution predictions on deleted-class test data; predictive entropy is measured in natural-log units (nats); a linear probe trains a *new* decoder on frozen features of deleted train images, then tests on deleted test images. Student-own and reference-own attack AUCs use **retained** training/test membership, not deleted-member/nonmember evidence.
- **Time**: 600 minutes is only a soft safety budget in this notebook, **not** extra Kaggle GPU hours. Save notebook output and attach it on the next session to recover checkpoints. Check `TRAIN_SEED`, `NORMALIZE_*`, model definitions, and any randomness before a final reported study.
- **Do not claim successful forgetting from high JSD/entropy or an AUC near 0.5 alone.** Interpret these alongside utility, confidence, and probe results with matched data and multiple seeds if feasible.